# Spurious-detection cleaning: photutils versus SEP

This notebook compares the photutils `get_spurious_labels` function
with the cleaning step of SEP (and therefore SourceExtractor), whose
CLEAN test it implements. Each section isolates one place where SEP
departs from the algorithm it describes, drives SEP into that behavior
with a synthetic scene, and shows why the photutils result is the
correct one.

The scenes are noiseless images of Gaussian sources, so every number
is reproducible. SEP is run without a convolution kernel and with a
single deblending level, which matches `detect_sources` on the same
image pixel for pixel. SEP is run twice per scene, without and with
cleaning, and the labels whose pixels vanish from the cleaned
segmentation map are the sources SEP removed. SEP does not run its
cleaning step on a user-supplied segmentation map, so every SEP result
here comes from SEP's own detection of the same scene.

Requires the optional `sep` package. Run from the `benchmarks`
directory so that the halo field of `bench_clean_sep.py` can be
imported.

In [ ]:
# Licensed under a 3-clause BSD style license - see LICENSE.rst
import matplotlib.pyplot as plt
import numpy as np
import sep
from astropy.modeling.models import Gaussian2D
from astropy.visualization import simple_norm
from matplotlib.colors import ListedColormap

from bench_clean_sep import make_halo_image
from photutils.segmentation import (SegmentationImage, detect_sources,
                                    get_spurious_labels)
# Private helpers, used only to display the intermediate quantities
from photutils.segmentation.clean import (CLEAN_ZONE, _measure_segments,
                                          _wing_model)

THRESHOLD = 5.0
N_PIXELS = 4
CLEAN_PARAM = 1.0

plt.rcParams['figure.dpi'] = 100
plt.rcParams['image.origin'] = 'lower'
plt.rcParams['image.interpolation'] = 'nearest'

In [ ]:
def make_scene(sources, shape=(100, 60)):
    """Return a noiseless image of (amplitude, x, y, sigma) Gaussians."""
    yy, xx = np.mgrid[0:shape[0], 0:shape[1]]
    data = np.zeros(shape)
    for amplitude, x, y, sigma in sources:
        data += Gaussian2D(amplitude, x, y, sigma, sigma)(xx, yy)
    return data


def run_sep(data, *, clean, threshold=THRESHOLD, n_pixels=N_PIXELS):
    """Run sep.extract with no kernel and no deblending."""
    _, segmap = sep.extract(np.ascontiguousarray(data), threshold,
                            minarea=n_pixels, filter_kernel=None,
                            deblend_nthresh=1, clean=clean,
                            clean_param=CLEAN_PARAM, segmentation_map=True)
    return segmap


def sep_cleaning(data, **kwargs):
    """Return SEP's uncleaned map, cleaned map, and removed labels."""
    segmap = run_sep(data, clean=False, **kwargs)
    cleaned = run_sep(data, clean=True, **kwargs)
    removed = np.unique(segmap[(segmap > 0) & (cleaned == 0)])
    return SegmentationImage(segmap.astype(int)), cleaned, removed


def photutils_cleaning(data, segm, threshold=THRESHOLD, n_pixels=N_PIXELS):
    """Return the get_spurious_labels table and the merged segments."""
    result = get_spurious_labels(data, segm, threshold, n_pixels,
                                 clean_param=CLEAN_PARAM)
    merged = segm.copy()
    for absorber in np.unique(result['absorbed_by']):
        labels = result['label'][result['absorbed_by'] == absorber]
        merged.reassign_labels(labels, absorber)
    return result, merged


def label_at(segm, source):
    """Return the label at the center of a source."""
    _, x, y, _ = source
    return int(segm.data[round(y), round(x)])


def show_image(ax, data, segm=None, title='', labels=True):
    """Show an image with the segment outlines and label numbers."""
    norm = simple_norm(data, 'sqrt', vmin=0, vmax=np.nanmax(data))
    ax.imshow(data, norm=norm, cmap='Greys_r')
    if segm is not None:
        segm.plot_patches(ax=ax, edgecolor='tab:orange', facecolor='none',
                          lw=1.2)
        if labels:
            for label, slc in zip(segm.labels, segm.slices, strict=True):
                yc = 0.5 * (slc[0].start + slc[0].stop)
                xc = 0.5 * (slc[1].start + slc[1].stop)
                ax.text(xc + 4, yc, str(label), color='tab:orange',
                        fontsize=11, va='center',
                        bbox={'facecolor': 'black', 'alpha': 0.6, 'pad': 1,
                              'lw': 0})
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


def show_segmap(ax, segmap, title=''):
    """Show a segmentation map with zero in black."""
    n_labels = int(segmap.max())
    colors = plt.get_cmap('tab10')(np.arange(max(n_labels, 1)) % 10)
    cmap = ListedColormap(np.vstack(([[0, 0, 0, 1]], colors)))
    ax.imshow(segmap, cmap=cmap, vmin=-0.5, vmax=n_labels + 0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])


def pair_diagnostics(data, segm, threshold=THRESHOLD, n_pixels=N_PIXELS):
    """Return the per-source quantities and pairwise helper functions."""
    props = _measure_segments(data, data, segm,
                              np.asarray(threshold, float), n_pixels)
    index = {int(label): k for k, label in enumerate(props['label'])}

    def wing(absorber, victim):
        e = np.array([index[absorber]])
        v = np.array([index[victim]])
        return float(_wing_model(props, e, v, CLEAN_PARAM)[0])

    def level(label):
        return float(props['mthresh'][index[label]])

    def separation(label1, label2):
        k1, k2 = index[label1], index[label2]
        dist = np.hypot(props['x'][k1] - props['x'][k2],
                        props['y'][k1] - props['y'][k2])
        zone = CLEAN_ZONE * (props['a'][k1] + props['a'][k2])
        return dist, zone

    return props, wing, level, separation

## 1. The CLEAN test

CLEAN describes every detected source by a Moffat-like wing model
built from its isophotal measurements,

$$ I(r) = A \, (1 + \alpha r^2)^{-\beta}, $$

where $\beta$ is `CLEAN_PARAM`, $A$ comes from the source flux and
its ellipse area, $r$ is measured in the source's own elliptical
metric, and $\alpha$ is chosen so that the model falls to the
detection threshold at the source's isophotal extent. For each pair of
sources whose centroids lie within ten times the sum of their
semimajor axes, the wing model of the brighter source is evaluated at
the centroid of the fainter one. If it exceeds the height of the
fainter source's `n_pixels`-th brightest pixel above the threshold
(its *comparison level*), the fainter source could not have been
detected on its own and is spurious.

The scene below is a bright star with a faint blob 18 pixels away.
The blob is its own segment, but the star's wing model at the blob's
centroid exceeds the blob's comparison level.

In [ ]:
STAR = (1000.0, 30.0, 70.0, 3.5)
NEAR_BLOB = (7.0, 30.0, 52.0, 2.5)

data = make_scene([STAR, NEAR_BLOB])
segm = detect_sources(data, THRESHOLD, N_PIXELS)
star = label_at(segm, STAR)
blob = label_at(segm, NEAR_BLOB)
props, wing, level, separation = pair_diagnostics(data, segm)

# The star's wing model along the column through both centroids
k = int(np.flatnonzero(props['label'] == star)[0])
unit_area = np.pi * props['a'][k] * props['b'][k]
amp = props['flux'][k] / (2 * unit_area * props['abcor'][k])
alpha = (((amp / props['thresh'][k]) ** (1 / CLEAN_PARAM) - 1)
         * unit_area / props['npix'][k])
yy = np.arange(data.shape[0])
dy = yy - props['y'][k]
model = amp * (1 + alpha * props['cyy'][k] * dy**2) ** (-CLEAN_PARAM)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.semilogy(yy, data[:, 30], color='k',
            label='image, column through both sources')
ax.semilogy(yy, model, color='tab:red', label="star's wing model")
ax.axhline(THRESHOLD, color='0.6', ls='--',
           label='detection threshold')
yb = props['y'][int(np.flatnonzero(props['label'] == blob)[0])]
ax.plot([yb], [THRESHOLD + level(blob)], marker='o', color='tab:blue',
        ls='none',
        label=f"blob's comparison level (threshold + {level(blob):.2f})")
ax.plot([yb], [wing(star, blob)], marker='s', color='tab:red',
        ls='none', label=f'wing model at the blob ({wing(star, blob):.2f})')
ax.set_ylim(0.05, 2000)
ax.set_xlabel('y (pixels)')
ax.set_ylabel('value')
ax.legend(fontsize=9, loc='upper left')
ax.set_title('The star wing model at the blob exceeds the blob level, '
             'so the blob is spurious')
plt.show()

print(f'wing model at the blob: {wing(star, blob):.3f}   '
      f'blob comparison level: {level(blob):.3f}')
dist, zone = separation(star, blob)
print(f'separation {dist:.1f} px, cleaning zone {zone:.1f} px')

## 2. Where the two codes agree, and what SEP does with the result

On this scene both codes flag the blob. The difference is what happens
next. SEP marks the blob as a non-survivor and then builds its catalog
and segmentation map from the survivors only, so the blob's pixels
become zero: the cleaned map has a hole where a real detection used to
be. SourceExtractor instead merges a cleaned object into the object
that absorbed it. Photutils returns the pairs and leaves the choice to
you. Merging with `reassign_labels` reproduces SourceExtractor, and
`remove_labels` reproduces SEP.

In [ ]:
sep_segm, sep_cleaned, sep_removed = sep_cleaning(data)
result, merged = photutils_cleaning(data, sep_segm)
print('SEP removed labels:', [int(v) for v in sep_removed])
print('photutils result:')
print(result)

fig, axes = plt.subplots(1, 4, figsize=(13, 5.5))
show_image(axes[0], data, sep_segm, 'image and SEP segments')
show_segmap(axes[1], sep_segm.data, 'SEP, before cleaning')
show_segmap(axes[2], sep_cleaned, 'SEP, after cleaning (hole)')
show_segmap(axes[3], merged.data, 'photutils, merged into absorber')
plt.tight_layout()
plt.show()

hole = np.count_nonzero((sep_segm.data > 0) & (sep_cleaned == 0))
print(f'{hole} pixels of a detected source are unassigned in the '
      'cleaned SEP map')

## 3. Bug: the heap that selects the comparison level

The comparison level is the height of the `minarea`-th brightest
pixel above the threshold. SEP (and SourceExtractor, from which the
code is copied verbatim) finds it with a min-heap of the `minarea`
largest values seen so far, whose root is the answer. The sift-down
that restores the heap after replacing the root continues from the
wrong node: after swapping with a child, it moves to the *sibling* of
that child instead of the child itself. The heap invariant breaks, and
the root can end up holding a value larger than the true minimum of
the retained set.

The function below is a line-for-line port of `analysemthresh` in
`sep/src/analyse.c`. The `fixed` flag corrects only the descent index.

In [ ]:
def sep_comparison_level(excess, minarea, *, fixed=False):
    """Port of SEP's analysemthresh heap (fixed=True corrects the descent)."""
    heap = [0.0] * minarea
    h = minarea
    heapt = 0
    for tpix in excess:
        tpix = float(np.float32(tpix))
        if h > 0:
            heap[heapt] = tpix
            heapt += 1
        elif h:
            if tpix > heap[0]:
                heap[0] = tpix
                j = 0
                while True:
                    k = (j + 1) << 1
                    if k > minarea:
                        break
                    hk = k
                    if k != minarea and heap[hk - 1] > heap[hk]:
                        hk += 1
                        k += 1
                    hk -= 1
                    if heap[j] <= heap[hk]:
                        break
                    heap[hk], heap[j] = heap[j], heap[hk]
                    j = hk if fixed else k
        else:
            heap.sort()
        h -= 1
    return heap[0]

The halo field of `bench_clean_sep.py` (bright stars with faint
sources scattered in their wings, plus noise) contains a source that
SEP keeps and photutils flags. Its pixel values, taken in raster order
as SEP's pixel list stores them for an object detected in one pass,
reproduce SEP's level exactly. The heap as written returns a value
well above the true fifth-brightest pixel, above the absorber's wing
at that point, so SEP keeps a source that its own test says is
spurious.

In [ ]:
halo = make_halo_image(1500, seed=0)
sep.set_extract_pixstack(max(sep.get_extract_pixstack(), halo.size))
HALO_THRESHOLD, HALO_N_PIXELS = 5.0, 5
halo_segm, halo_cleaned, halo_removed = sep_cleaning(
    halo, threshold=HALO_THRESHOLD, n_pixels=HALO_N_PIXELS)
halo_result = get_spurious_labels(halo, halo_segm, HALO_THRESHOLD,
                                  HALO_N_PIXELS)
photutils_only = np.setdiff1d(halo_result['label'], halo_removed)
sep_only = np.setdiff1d(halo_removed, halo_result['label'])
print(f'{halo_segm.n_labels} sources, SEP removed {len(halo_removed)}, '
      f'photutils flagged {len(halo_result)}. Flagged only by photutils: '
      f'{[int(v) for v in photutils_only]}, removed only by SEP: '
      f'{[int(v) for v in sep_only]}')

victim = int(photutils_only[0])
absorber = int(halo_result['absorbed_by'][halo_result['label']
                                          == victim][0])
props, wing, level, separation = pair_diagnostics(
    halo, halo_segm, HALO_THRESHOLD, HALO_N_PIXELS)

pixels = np.flatnonzero(halo_segm.data.ravel() == victim)  # raster order
excess = halo.ravel()[pixels] - HALO_THRESHOLD
exact = np.sort(excess)[::-1][HALO_N_PIXELS - 1]
buggy = sep_comparison_level(excess, HALO_N_PIXELS)
fixed = sep_comparison_level(excess, HALO_N_PIXELS, fixed=True)
model = wing(absorber, victim)
print(f'true {HALO_N_PIXELS}th brightest pixel above threshold: '
      f'{exact:.4f}')
print(f'SEP heap as written:                          {buggy:.4f}')
print(f'SEP heap with the descent fixed:              {fixed:.4f}')
print(f"absorber's wing model at the victim:          {model:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5),
                         gridspec_kw={'width_ratios': [1, 1.6]})

# Cutout around the victim and its absorber
slc = halo_segm.slices[list(halo_segm.labels).index(victim)]
k_e = int(np.flatnonzero(props['label'] == absorber)[0])
k_v = int(np.flatnonzero(props['label'] == victim)[0])
x0 = int(min(props['x'][k_e], props['x'][k_v])) - 25
x1 = int(max(props['x'][k_e], props['x'][k_v])) + 25
y0 = int(min(props['y'][k_e], props['y'][k_v])) - 25
y1 = int(max(props['y'][k_e], props['y'][k_v])) + 25
cut = halo[y0:y1, x0:x1]
norm = simple_norm(cut, 'log', vmin=1, vmax=cut.max())
axes[0].imshow(cut, norm=norm, cmap='Greys_r', extent=(x0, x1, y0, y1))
halo_segm.plot_patches(ax=axes[0], labels=[victim, absorber],
                       edgecolor='tab:orange', facecolor='none', lw=1.2)
axes[0].plot(props['x'][k_v], props['y'][k_v], 'x', color='tab:blue',
             ms=9, label=f'victim {victim}')
axes[0].plot(props['x'][k_e], props['y'][k_e], '+', color='tab:red',
             ms=12, label=f'absorber {absorber}')
axes[0].set_xlim(x0, x1)
axes[0].set_ylim(y0, y1)
axes[0].legend(fontsize=9, loc='upper left')
axes[0].set_title('halo field cutout (log stretch)')

order = np.argsort(excess)[::-1]
axes[1].bar(np.arange(len(excess)), excess[order], color='0.7',
            label='victim pixels, sorted')
axes[1].bar([HALO_N_PIXELS - 1], [exact], color='tab:blue',
            label=f'{HALO_N_PIXELS}th brightest (photutils level)')
axes[1].axhline(buggy, color='tab:red', ls='-',
                label=f'SEP heap as written ({buggy:.2f})')
axes[1].axhline(model, color='tab:green', ls='--',
                label=f"absorber's wing model ({model:.2f})")
axes[1].set_xlabel('pixel rank')
axes[1].set_ylabel('height above threshold')
axes[1].legend(fontsize=9)
axes[1].set_title('SEP level > wing > true level: '
                  'SEP keeps a spurious source')
plt.tight_layout()
plt.show()

The wing model sits between the two levels. With the true level the
source is spurious. With SEP's inflated level it survives. The
photutils level is the exact order statistic, so photutils flags it.
Everything else about this pair agrees between the two codes to
float32 precision (the benchmark script checks the centroids, axes,
ellipse coefficients, fluxes, and areas).

## 4. Quirk: the pixel that is never inserted

The same heap has a second flaw. The first `minarea` pixels fill the
array, and the next pixel triggers the sort that turns the array into
a heap, but that pixel is never inserted. If it is one of the
`minarea` brightest pixels of the object, the level drops to the
next-brightest pixel, in the opposite direction from the sift-down
bug. The effect is largest for an object with exactly `minarea + 1`
pixels, where the level becomes the faintest of the first `minarea`
pixels in the list.

The halo field contains such a case: the source that SEP removes and
photutils keeps. It has six pixels, `minarea` is five, and the sixth
pixel in raster order is its brightest. SEP's level is the faintest
of the other five, twenty times too low, and the wing of a survivor 44
pixels away is enough to remove it.

In [ ]:
victim2 = int(sep_only[0])
k_v = int(np.flatnonzero(props['label'] == victim2)[0])
pixels = np.flatnonzero(halo_segm.data.ravel() == victim2)  # raster order
excess2 = halo.ravel()[pixels] - HALO_THRESHOLD
exact2 = np.sort(excess2)[::-1][HALO_N_PIXELS - 1]
sep_level2 = sep_comparison_level(excess2, HALO_N_PIXELS)

# The brighter source within the cleaning zone whose wing is highest
# at the victim
candidates = []
for label in props['label']:
    label = int(label)
    k = int(np.flatnonzero(props['label'] == label)[0])
    if label == victim2 or props['flux'][k] < props['flux'][k_v]:
        continue
    dist, zone = separation(label, victim2)
    if dist <= zone:
        candidates.append((wing(label, victim2), label, dist))
model2, absorber2, dist2 = max(candidates)
print(f'source {victim2}: {len(excess2)} pixels, minarea {HALO_N_PIXELS}')
print(f'true {HALO_N_PIXELS}th brightest pixel above threshold: '
      f'{exact2:.4f}')
print(f'SEP heap (pixel {HALO_N_PIXELS + 1} skipped):            '
      f'{sep_level2:.4f}')
print(f'highest wing model at the source, from {absorber2} at '
      f'{dist2:.0f} px: {model2:.4f}')
print(f'SEP: removed (level {sep_level2:.3f} < wing {model2:.3f}). '
      f'photutils: kept (level {exact2:.3f} > wing {model2:.3f})')

fig, ax = plt.subplots(figsize=(8, 3.8))
colors = ['0.7'] * len(excess2)
colors[HALO_N_PIXELS] = 'tab:red'
ax.bar(np.arange(len(excess2)), excess2, color=colors)
ax.axhline(exact2, color='tab:blue',
           label=f'true {HALO_N_PIXELS}th brightest ({exact2:.3f}, '
                 'photutils level)')
ax.axhline(sep_level2, color='tab:red', ls='--',
           label=f'SEP heap, pixel {HALO_N_PIXELS + 1} skipped '
                 f'({sep_level2:.3f})')
ax.axhline(model2, color='tab:green', ls=':',
           label=f'wing model of source {absorber2} ({model2:.3f})')
ax.set_xticks(np.arange(len(excess2)))
ax.set_xticklabels(np.arange(1, len(excess2) + 1))
ax.set_xlabel('pixel position in raster order')
ax.set_ylabel('height above threshold')
ax.legend(fontsize=9)
ax.set_title(f'Source {victim2}: its brightest pixel is the one SEP '
             'never inserts')
plt.tight_layout()
plt.show()

The two heap flaws push the level in opposite directions, and in this
field each one flips exactly one decision: one spurious source kept,
one real source removed. Photutils uses the exact order statistic and
gets both right.

## 5. Quirk: the answer depends on the label order

SEP tests the pairs in a flat double loop over the objects in label
order. Two consequences follow. A source that is absorbed partway
through its own inner loop keeps absorbing later sources, and a source
that is already absorbed when its loop begins is skipped entirely,
even though nothing about the physics has changed. Labels are assigned
in raster order, so flipping the image vertically reverses the label
order and can change SEP's answer.

The scene has a bright star, a broad faint blob 30 pixels away that
the star absorbs, and a very faint blob 18 pixels beyond it. The very
faint blob is inside the broad blob's reach but outside the star's
cleaning zone.

In [ ]:
BROAD = (5.5, 30.0, 40.0, 6.0)
VERY_FAINT = (5.3, 30.0, 22.0, 3.0)
scene = make_scene([STAR, BROAD, VERY_FAINT])
flipped = np.ascontiguousarray(scene[::-1])
flipped_sources = [(a, x, scene.shape[0] - 1 - y, s)
                   for a, x, y, s in (STAR, BROAD, VERY_FAINT)]

fig, axes = plt.subplots(2, 3, figsize=(11, 8))
for row, (name, image, sources) in enumerate(
        (('as detected', scene, (STAR, BROAD, VERY_FAINT)),
         ('flipped vertically', flipped, flipped_sources))):
    sep_segm, sep_cleaned, sep_removed = sep_cleaning(image)
    result, merged = photutils_cleaning(image, sep_segm)
    names = dict(zip((label_at(sep_segm, s) for s in sources),
                     ('star', 'broad blob', 'very faint blob'),
                     strict=True))
    print(f'{name}: labels {names}')
    flagged = ', '.join(f'{lab} (absorbed by {ab})'
                        for lab, ab in zip(result['label'],
                                           result['absorbed_by'],
                                           strict=True))
    print(f'   SEP removed {[int(v) for v in sep_removed]}, '
          f'photutils flagged {flagged}')
    show_image(axes[row, 0], image, sep_segm, f'{name}: SEP segments')
    show_segmap(axes[row, 1], sep_cleaned, f'{name}: SEP cleaned')
    show_segmap(axes[row, 2], merged.data,
                f'{name}: photutils merged')
plt.tight_layout()
plt.show()

SEP gives two different answers for the same scene. In the original
orientation the very faint blob is labeled first, the broad blob
absorbs it, and then the star absorbs the broad blob, so a source has
been removed by an absorber that does not itself survive. In the
flipped orientation the star is labeled first and absorbs the broad
blob, the broad blob is skipped as a dead source, and the very faint
blob survives.

Photutils gives the same answer in both orientations. It resolves the
sources in order of decreasing flux, and a source counts as spurious
only if the wing of a brighter *surviving* source exceeds its level.
The broad blob is absorbed by the star, so it cannot absorb anything,
and the star's own wing does not reach the very faint blob. The
pairwise quantities below show that this is the only consistent
reading of the test: the star does reach the broad blob, and the broad
blob would reach the very faint blob, but the star does not.

In [ ]:
sep_segm, _, _ = sep_cleaning(scene)
props, wing, level, separation = pair_diagnostics(scene, sep_segm)
names = {label_at(sep_segm, s): n
         for s, n in zip((STAR, BROAD, VERY_FAINT),
                         ('star', 'broad blob', 'very faint blob'),
                         strict=True)}
print(f'{"absorber":>16} {"victim":>16} {"wing model":>11} {"level":>8} '
      f'{"separation":>11} {"zone":>7}  reaches?')
for a, v in ((STAR, BROAD), (STAR, VERY_FAINT), (BROAD, VERY_FAINT)):
    la, lv = label_at(sep_segm, a), label_at(sep_segm, v)
    dist, zone = separation(la, lv)
    reaches = wing(la, lv) > level(lv) and dist <= zone
    print(f'{names[la]:>16} {names[lv]:>16} {wing(la, lv):11.3f} '
          f'{level(lv):8.3f} {dist:11.1f} {zone:7.1f}  {reaches}')

## 6. Quirk: the area correction ignores per-pixel thresholds

SourceExtractor supports a spatially varying detection threshold. Its
comparison level uses each pixel's own threshold, but the pixel counts
that feed the wing model's area correction (the number of pixels above
the threshold, and above the level midway to the peak) use the mean
threshold over the object. A pixel that is below its own detection
threshold is therefore counted as "above threshold", and the
correction that scales the model amplitude sees a profile that was
never detected. Photutils compares each pixel with its own threshold
in those counts too.

The scene below is a star with a broad halo, whose area correction
sits below its cap of 1 so that the counts actually matter, and a
faint blob at the edge of its reach. The threshold is raised on the
left half of the star's segment and lowered on the right half so
that the mean over the segment is unchanged. With the same mean, the
model normalization and the blob's own level are identical for both
rules, so the only difference is the per-pixel counts.

In [ ]:
CORE = (40.0, 30.0, 70.0, 1.0)
HALO = (7.0, 30.0, 70.0, 12.0)
EDGE_BLOB = (6.0, 30.0, 24.0, 2.0)
halo_scene = make_scene([CORE, HALO, EDGE_BLOB], shape=(120, 60))
halo_segm2 = detect_sources(halo_scene, THRESHOLD, N_PIXELS)
star2 = label_at(halo_segm2, CORE)
blob2 = label_at(halo_segm2, EDGE_BLOB)

_, xx = np.mgrid[0:120, 0:60]
left = (halo_segm2.data == star2) & (xx < 30)
right = (halo_segm2.data == star2) & (xx >= 30)
threshold_map = np.full(halo_scene.shape, THRESHOLD)
threshold_map[left] += 4.5
threshold_map[right] -= 4.5 * np.count_nonzero(left) / np.count_nonzero(right)
mean_threshold = threshold_map[halo_segm2.data == star2].mean()
print(f'mean threshold over the star segment: {mean_threshold:.4f}')

# SourceExtractor's rule counts pixels against the mean threshold,
# which for this threshold map is identical to the uniform threshold
_, wing_mean, level_mean, _ = pair_diagnostics(halo_scene, halo_segm2,
                                                THRESHOLD)
_, wing_pixel, level_pixel, _ = pair_diagnostics(halo_scene, halo_segm2,
                                                  threshold_map)
rules = (('SourceExtractor rule (mean threshold in the counts)',
          wing_mean, level_mean),
         ('photutils rule (each pixel against its own threshold)',
          wing_pixel, level_pixel))
for name, wing_fn, level_fn in rules:
    model = wing_fn(star2, blob2)
    verdict = 'spurious' if model > level_fn(blob2) else 'survives'
    print(f'{name}:')
    print(f'    wing model {model:.4f}, blob level {level_fn(blob2):.4f} '
          f'-> blob {verdict}')

for name, thr in (('uniform threshold', THRESHOLD),
                  ('threshold map', threshold_map)):
    result = get_spurious_labels(halo_scene, halo_segm2, thr, N_PIXELS)
    pairs = dict(zip(result['label'].tolist(),
                     result['absorbed_by'].tolist(), strict=True))
    print(f'photutils, {name}: {pairs}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5.5))
show_image(axes[0], halo_scene, halo_segm2,
           'star with halo, and a blob at the edge of its reach')
im = axes[1].imshow(np.where(halo_segm2.data == star2, threshold_map,
                             np.nan), cmap='coolwarm')
halo_segm2.plot_patches(ax=axes[1], edgecolor='0.3', facecolor='none',
                        lw=1)
axes[1].set_title('threshold map over the star segment')
axes[1].set_xticks([])
axes[1].set_yticks([])
plt.colorbar(im, ax=axes[1], fraction=0.05, label='threshold')

star_values = halo_scene[halo_segm2.data == star2]
star_thresh = threshold_map[halo_segm2.data == star2]
axes[2].scatter(star_thresh, star_values, s=8, color='0.5',
                label='star pixels')
above_own = star_values > star_thresh
axes[2].scatter(star_thresh[~above_own], star_values[~above_own], s=14,
                color='tab:red',
                label='below own threshold, counted by SourceExtractor')
axes[2].axvline(THRESHOLD, color='k', ls='--', label='mean threshold')
axes[2].set_yscale('log')
axes[2].set_xlabel('pixel threshold')
axes[2].set_ylabel('pixel value')
axes[2].legend(fontsize=8, loc='center')
axes[2].set_title('pixels below their own threshold')
plt.tight_layout()
plt.show()

Under SourceExtractor's rule the blob is spurious, under the
per-pixel rule it survives, and the only quantity that changed is the
count of star pixels that are genuinely above the threshold that
detected them. Those pixels shown in red were never detections, and
counting them inflates the isophotal area drop that the correction
measures.

## 7. The wing model is elliptical, not circular

The radial argument of the wing model is the quadratic form
$C_{xx}\,\Delta x^2 + C_{yy}\,\Delta y^2 + C_{xy}\,\Delta x\,\Delta y$
built from the source's own second moments, so the model contours
have the same shape and orientation as the source. This is the
SourceExtractor definition and photutils follows it, so the two codes
agree here. It matters for elongated sources: the wing reaches much
farther along the major axis than the minor axis, by the square of
the axis ratio.

The scene is a star elongated four to one, with two identical faint
blobs 35 pixels away, one on each axis. The blob on the major axis is
absorbed and the one on the minor axis is kept.

In [ ]:
yy2, xx2 = np.mgrid[0:140, 0:140]
elongated = (Gaussian2D(1000, 70, 70, 7.0, 1.75)(xx2, yy2)
             + Gaussian2D(7, 105, 70, 2.0, 2.0)(xx2, yy2)
             + Gaussian2D(7, 70, 105, 2.0, 2.0)(xx2, yy2))
sep_segm_e, sep_cleaned_e, sep_removed_e = sep_cleaning(elongated)
result_e, merged_e = photutils_cleaning(elongated, sep_segm_e)
star_e = int(sep_segm_e.data[70, 70])
major_e = int(sep_segm_e.data[70, 105])
minor_e = int(sep_segm_e.data[105, 70])
props_e, wing_e, level_e, _ = pair_diagnostics(elongated, sep_segm_e)
k = int(np.flatnonzero(props_e['label'] == star_e)[0])
print(f"star axes a={props_e['a'][k]:.2f} b={props_e['b'][k]:.2f}")
for name, lab in (('major-axis blob', major_e), ('minor-axis blob', minor_e)):
    print(f'{name}: wing model {wing_e(star_e, lab):.3f}, '
          f'level {level_e(lab):.3f}')
print('SEP removed', [int(v) for v in sep_removed_e],
      '  photutils flagged', result_e['label'].tolist(),
      f'  (major-axis blob = {major_e}, minor-axis blob = {minor_e})')

# The star's wing model over the whole image
unit_area = np.pi * props_e['a'][k] * props_e['b'][k]
amp_e = props_e['flux'][k] / (2 * unit_area * props_e['abcor'][k])
alpha_e = (((amp_e / props_e['thresh'][k]) ** (1 / CLEAN_PARAM) - 1)
           * unit_area / props_e['npix'][k])
dx = xx2 - props_e['x'][k]
dy = yy2 - props_e['y'][k]
r2 = (props_e['cxx'][k] * dx**2 + props_e['cyy'][k] * dy**2
      + props_e['cxy'][k] * dx * dy)
model_map = amp_e * (1 + alpha_e * r2) ** (-CLEAN_PARAM)

fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                         gridspec_kw={'width_ratios': [1.3, 1]})
show_image(axes[0], elongated, sep_segm_e,
           'elongated star and two blobs, with the star wing model')
levels = [0.25, 0.5, level_e(major_e), 2.0, 4.0]
cs = axes[0].contour(model_map, levels=sorted(levels), colors='tab:red',
                     linewidths=[0.8, 0.8, 1.8, 0.8, 0.8])
axes[0].clabel(cs, fmt='%.2f', fontsize=8)
axes[0].text(2, 2, 'thick contour: wing model = blob level',
             color='tab:red', fontsize=9)

names = ['major-axis blob', 'minor-axis blob']
wings = [wing_e(star_e, major_e), wing_e(star_e, minor_e)]
levels_b = [level_e(major_e), level_e(minor_e)]
pos = np.arange(2)
axes[1].bar(pos - 0.18, wings, width=0.36, color='tab:red',
            label='star wing model at the blob')
axes[1].bar(pos + 0.18, levels_b, width=0.36, color='tab:blue',
            label='blob comparison level')
axes[1].set_xticks(pos)
axes[1].set_xticklabels(names)
axes[1].set_ylabel('height above threshold')
axes[1].legend(fontsize=9)
axes[1].set_title('same distance, different wing')
plt.tight_layout()
plt.show()

## 8. Minor difference: float32 comparisons

SEP casts the wing model to float32 before comparing it with the
comparison level, which it also stores as float32. Photutils compares
both in float64. This can only change a decision when the model and
the level agree to about one part in ten million, so it has no visual
example here. It is listed for completeness.

## Summary

| Behavior in SEP | Effect | photutils |
|---|---|---|
| Heap sift-down descends into the wrong node | comparison level too high, spurious sources kept | exact order statistic |
| The `minarea + 1`-th pixel is never inserted into the heap | comparison level too low when that pixel is bright | exact order statistic |
| Pairs tested in label order, absorbed sources keep absorbing | result depends on label order, sources removed by non-survivors | resolved by decreasing flux, only survivors absorb, order independent |
| Area-correction counts use the object's mean threshold | pixels below their own detection threshold counted as detections | each pixel compared with its own threshold |
| Cleaned objects dropped from the map | holes in the segmentation image | pairs returned, merge or drop is the caller's choice |
| Model and level compared in float32 | negligible | float64 |

Everything else, the wing model, its normalization, the area
correction, the cleaning zone, and the treatment of segments with too
few pixels, is the CLEAN algorithm as SourceExtractor defines it, and
the benchmark script confirms that the two codes agree to float32
precision on every per-source quantity.